# Physics-guided histopathology experiments (UBC-OCEAN) - Kaggle runner

**One-time setup**
1. Kaggle account with a **verified phone** (Settings > Phone verification). This unlocks GPU and Internet.
2. *Create > New Dataset*, name it `physics-paper`, **Private**, and upload: `ubc_patches.zip`, the 6 code files
   (`ovarian_pipeline.py, novelty_utils.py, novel_model.py, histo_utils.py, histo_model.py, test_histo_utils.py`)
   and, optionally, your finished Drive folders `results_main` and `results_rest` (they are small; they let the runs continue instead of restarting).
3. *Create > New Notebook*, then *File > Import notebook* and pick this file. In the right-hand panel: **Add Input** > your dataset;
   **Session options > Accelerator = GPU T4 x2**; **Internet = On** (needed once to download the EfficientNet weights).
4. Run the cells in order. For the long runs use *Save Version > Save & Run All (Commit)* so it keeps going with the tab closed.

## 1. Settings and GPU check

In [ ]:
import os, subprocess, torch
RUN_TESTS = True       # unit tests + tiny smoke run (a few minutes)
RUN_STATS = True       # section 5: A_baseline, G_colorjitter, H_stainaug, E_full with slide predictions (most important for the paper)
RUN_REST  = True       # section 7: remaining ablations B, C, D, F
print("torch", torch.__version__, "| GPU available:", torch.cuda.is_available())
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout)
assert torch.cuda.is_available(), "No GPU: Session options > Accelerator > GPU T4 x2, then re-run"

## 2. Locate the dataset, copy code, unpack patches, restore earlier results

In [ ]:
import os, shutil, zipfile, time, glob
INPUT, WORK, CODE, TMP = "/kaggle/input", "/kaggle/working", "/kaggle/working/code", "/kaggle/temp"
os.makedirs(CODE, exist_ok=True); os.makedirs(TMP, exist_ok=True)

import re
def find_file(name):
    # accepts "histo_model.py" as well as browser-renamed copies such as "histo_model (1).py"
    stem, ext = os.path.splitext(name)
    pat = re.compile(rf"^{re.escape(stem)}( \((\d+)\))?{re.escape(ext)}$")
    hits = []
    for dp, _, fs in os.walk(INPUT):
        for f in fs:
            m = pat.match(f)
            if m:
                hits.append((int(m.group(2) or 0), os.path.join(dp, f)))
    if not hits:
        raise FileNotFoundError(f"{name} not found under {INPUT} - did you add the dataset as Input?")
    return max(hits)[1]                      # highest "(n)" = newest download

for f in ["ovarian_pipeline.py", "novelty_utils.py", "novel_model.py",
          "histo_utils.py", "histo_model.py", "test_histo_utils.py"]:
    src = find_file(f)
    shutil.copy(src, f"{CODE}/{f}")           # always saved under the canonical name
    print("code:", os.path.basename(src), "->", f)
assert "--resume" in open(f"{CODE}/histo_model.py").read(), "old histo_model.py: upload the latest version"

t = time.time()
UBC = f"{TMP}/ubc"
zips = [os.path.join(dp, f) for dp, _, fs in os.walk(INPUT) for f in fs if f == "ubc_patches.zip"]
if zips:                                              # dataset kept the zip as is
    with zipfile.ZipFile(zips[0]) as z:
        z.extractall(TMP)
else:                                                 # Kaggle already unzipped it: copy the folder that holds patches/
    man = find_file("patch_manifest.csv")
    src = os.path.dirname(os.path.dirname(man))
    shutil.copytree(src, UBC, dirs_exist_ok=True)
assert os.path.exists(f"{UBC}/patches/patch_manifest.csv"), f"unexpected layout under {UBC}"
n = sum(len(fs) for _, _, fs in os.walk(UBC))
print(f"patches ready: {n} files in {time.time() - t:.0f}s  ->  {UBC}")

# earlier results (optional): copied so that --resume can continue them
for d in ["results_main", "results_rest", "results_stats"]:
    for dp, dn, _ in os.walk(INPUT):
        if d in dn:
            shutil.copytree(os.path.join(dp, d), f"{WORK}/{d}", dirs_exist_ok=True)
            print("restored earlier", d, "->", sorted(os.listdir(f"{WORK}/{d}"))[:6])
            break
os.chdir(CODE)

## 3. Unit tests and smoke tests (expect `ALL TESTS PASSED`, then a small results table)

In [ ]:
if RUN_TESTS:
    !python -W ignore test_histo_utils.py 2>&1 | tail -5
    !python histo_model.py --ubc-dir {UBC} --max-patches-per-slide 4 --folds 3 --epochs 2 --variants A_baseline E_full --workers 2 --dl-workers 2 --out /kaggle/temp/check_histo 2>&1 | tail -25

## 4. (Information) earlier main run
`results_main` (A_baseline, E_full, G_colorjitter) is already finished on Drive. If you uploaded it in the dataset it was restored in section 2 and is used by the merge in section 8. Nothing to run here.

## 5. Stronger baseline + saved slide predictions (most important run)
A_baseline, G_colorjitter, H_stainaug, E_full with identical folds, slide predictions for bootstrap CIs, and the gentler blur test.
Roughly 2 h on a T4 (estimate). `--resume` continues after a stop: just re-run the cell.

In [ ]:
OUT_STATS = f"{WORK}/results_stats"
os.makedirs(OUT_STATS, exist_ok=True)
if os.path.exists(f"{WORK}/results_main/radiomics_cache.npz") and not os.path.exists(f"{OUT_STATS}/radiomics_cache.npz"):
    shutil.copy(f"{WORK}/results_main/radiomics_cache.npz", OUT_STATS)
if RUN_STATS:
    !python histo_model.py --ubc-dir {UBC} --max-patches-per-slide 16 --folds 5 --epochs 10 --variants A_baseline G_colorjitter H_stainaug E_full --conditions clean clean_sub bright0.8 wb hscale1.3 escale0.7 blur blur0.7 jpeg30 noise --workers 2 --dl-workers 2 --resume --out {OUT_STATS}

## 6. Bootstrap confidence intervals and paired comparisons (needs section 5)

In [ ]:
import sys, json, numpy as np
sys.path.insert(0, CODE)
import histo_utils as hu
if os.path.exists(f"{OUT_STATS}/slide_predictions.csv"):
    sp = hu.load_slide_predictions(f"{OUT_STATS}/slide_predictions.csv")
    variants = [v for v in ["A_baseline", "G_colorjitter", "H_stainaug", "E_full"] if v in set(sp.variant)]
    text = "\n\n".join([hu.bootstrap_table_md(sp, variants, ref="A_baseline"),
                         hu.bootstrap_table_md(sp, variants, ref="G_colorjitter"),
                         hu.bootstrap_table_md(sp, variants, ref="H_stainaug", metric="macro_f1")])
    open(f"{WORK}/bootstrap_tables.md", "w", encoding="utf-8").write(text)
    print(text)
else:
    print("no slide_predictions.csv yet - run section 5 first")

## 7. Remaining ablation variants B, C, D, F
`results_rest` had finished about 3 of 5 folds on Colab. If you uploaded it to the dataset, `--resume` continues from there; otherwise it starts fresh (about 2-3 h).

In [ ]:
OUT_REST = f"{WORK}/results_rest"
if RUN_REST:
    !python histo_model.py --ubc-dir {UBC} --max-patches-per-slide 16 --folds 5 --epochs 10 --variants B_stainOD C_stainOD+consist D_attn_only F_consist_only --workers 2 --dl-workers 2 --resume --out {OUT_REST}

## 8. Merge main + rest into one ablation table, and collect the files to bring back

In [ ]:
res, cms = {}, {}
for d in [f"{WORK}/results_main", f"{WORK}/results_rest"]:
    if os.path.exists(f"{d}/results.json"):
        res.update(json.load(open(f"{d}/results.json")))
        if os.path.exists(f"{d}/confusion.json"):
            cms.update({k: np.array(v) for k, v in json.load(open(f"{d}/confusion.json")).items()})
order = ["A_baseline", "B_stainOD", "C_stainOD+consist", "D_attn_only", "F_consist_only", "G_colorjitter", "H_stainaug", "E_full"]
res = {k: res[k] for k in order if k in res}
cms = {k: cms[k] for k in order if k in cms}
BRING = f"{WORK}/bring_back"; os.makedirs(BRING, exist_ok=True)
if res:
    df = hu.summarize_histo(res)
    md_text = hu.to_markdown_histo(df, cms or None, hu.CLASSES if cms else None)
    open(f"{BRING}/tables_all.md", "w", encoding="utf-8").write(md_text)
    df.to_csv(f"{BRING}/summary_all_slide_level.csv", index=False)
    hu.plot_robustness_histo(df, f"{BRING}/robustness_all.png")
    print(md_text)
else:
    print("no finished results.json yet in results_main / results_rest")
for src in ["results_stats/tables.md", "results_stats/robustness.png", "results_stats/summary_slide_level.csv",
            "results_main/xai.json", "results_main/xai_examples.png", "bootstrap_tables.md"]:
    p = f"{WORK}/{src}"
    if os.path.exists(p):
        shutil.copy(p, f"{BRING}/{src.replace('/', '__')}")
print(sorted(os.listdir(BRING)))

## What to bring back
Download everything in `bring_back/` (Output tab on the right, after *Save Version*), or just paste the tables here.
The whole `results_*` folders are also saved as this notebook's output.